# FineTunePilot — Stage 3: OOM recovery demo on Kaggle

`finetunepilot run --force-oom` starts with batch size 16 and **skips** the VRAM pre-check, so
the first attempt runs out of GPU memory on purpose. The runner then classifies each error and
retries in a fresh process with the next fix:

1. halve batch size, double gradient accumulation (same effective batch)
2. turn on gradient checkpointing
3. cut `max_seq_length` by 25% (never below the longest row)

**Before you start:** Accelerator → **GPU T4**, **Internet on**. Takes roughly 5–15 minutes.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code and install it

In [ ]:
import os
%cd /kaggle/working
if not os.path.isdir("Finetune-pilot-ai"):
    !git clone --depth 1 --branch claude/zen-meitner-hetiul https://github.com/manojastro/Finetune-pilot-ai.git
%cd Finetune-pilot-ai
!pip install -q -e ".[gpu]"

## 3. One GPU only

In [ ]:
%env CUDA_VISIBLE_DEVICES=0

## 4. Normal run with the pre-check
The VRAM estimate should say the default config fits, and training should succeed on attempt 1.

In [ ]:
!COLUMNS=140 finetunepilot run --config configs/qwen05b_toolcall.yaml --max-attempts 4 --max-minutes 30

## 5. Forced OOM and recovery
Expect `CUDA_OOM` on the first attempt(s), then a success with a smaller batch and/or gradient checkpointing.

In [ ]:
!COLUMNS=140 finetunepilot run --config configs/qwen05b_toolcall.yaml --force-oom --max-attempts 4 --max-minutes 45

## 6. Copy this back into Claude Code
For both runs: the pre-check estimate, every attempt (error class, evidence, fix, duration) and the measured peak memory. Compare the estimate with `peak_gpu_memory_reserved_mb` to see how accurate the estimator is.

In [ ]:
import glob, json

def brief(path):
    log = json.load(open(path))
    pre = log.get("precheck", {})
    return {
        "run_id": log["run_id"], "status": log["status"], "stop_reason": log["stop_reason"],
        "duration_s": log["duration_s"], "force_oom": log["config"].get("force_oom"),
        "precheck": {k: pre.get(k) for k in ("status", "seq_len", "fixes")} | {
            "estimate_gb": (pre.get("estimate_after") or {}).get("total_gb"),
            "breakdown_gb": (pre.get("estimate_after") or {}).get("breakdown_gb"),
        },
        "attempts": [
            {k: a[k] for k in ("attempt", "status", "error_class", "evidence", "fix", "duration_s")}
            | {"peak_gpu_memory_reserved_mb": a["result"].get("peak_gpu_memory_reserved_mb"),
               "final_loss": a["result"].get("final_loss")}
            for a in log["attempts"]
        ],
        "hardware": log["hardware"], "library_versions": log["library_versions"],
    }

runs = sorted(p for p in glob.glob("runs/*.json") if "-dryrun-" not in p)
recovery = [p for p in runs if json.load(open(p)).get("stop_reason")][-2:]
print(json.dumps([brief(p) for p in recovery], indent=1))